In [2]:
import pandas as pd
import numpy as np
from datetime import datetime
import os

# =========================================
# CONFIGURATION
# =========================================

# Dataset paths
RAW_DATA_PATH = "C:\\Users\\sneha\\Desktop\\ecopackai\\Dataset"
PROCESSED_DATA_PATH = "C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\processed"

MATERIAL_FILE = "C:\\Users\\sneha\\Desktop\\ecopackai\\Dataset\\material_dataset (2).csv"
PRODUCT_FILE = "C:\\Users\\sneha\\Desktop\\ecopackai\\Dataset\\product_dataset (2).csv"

# Create processed data directory if it doesn't exist
os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)

print("✅ Setup complete.")


✅ Setup complete.


In [5]:
print("\n[1/6] Loading datasets...")

try:
    materials_df = pd.read_csv("C:\\Users\\sneha\\Desktop\\ecopackai\\Dataset\\material_dataset (2).csv")
    products_df = pd.read_csv("C:\\Users\\sneha\\Desktop\\ecopackai\\Dataset\\product_dataset (2).csv")
    print(f"✓ Materials loaded: {materials_df.shape}")
    print(f"✓ Products loaded: {products_df.shape}")
except FileNotFoundError as e:
    print(f"❌ Error: {e}")
    print("Please check your dataset paths.")



[1/6] Loading datasets...
✓ Materials loaded: (404, 23)
✓ Products loaded: (1500, 6)


In [6]:
print("\n[2/6] Exploring datasets...")

print("\nMaterial Dataset Info:")
print(f"  Rows: {len(materials_df)}")
print(f"  Columns: {len(materials_df.columns)}")
print(f"  Missing values: {materials_df.isnull().sum().sum()}")

print("\nProduct Dataset Info:")
print(f"  Rows: {len(products_df)}")
print(f"  Columns: {len(products_df.columns)}")
print(f"  Missing values: {products_df.isnull().sum().sum()}")



[2/6] Exploring datasets...

Material Dataset Info:
  Rows: 404
  Columns: 23
  Missing values: 15

Product Dataset Info:
  Rows: 1500
  Columns: 6
  Missing values: 0


In [7]:
print("\n[3/6] Creating product-material combinations...")

integrated_data = []

for _, product in products_df.iterrows():
    for _, material in materials_df.iterrows():
        record = {
            # Product Information
            'product_id': product['product_id'],
            'product_name': product['product_name'],
            'product_category': product['category'],
            'product_weight_kg': product['product_weight_kg'],
            'fragility_index': product['fragility_index'],
            'shipping_type': product['shipping_type'],
            
            # Material Information
            'material_id': material['Material ID'],
            'material_type': material['Material Type'],
            'packaging_type': material['Packaging Type'],
            'suitable_categories': material['Suitable Product Categories'],
            'recyclability_percent': material['Recyclability (%)'],
            'recyclability_category': material['Recyclability Category'],
            'biodegradation_days': material['Biodegradation Time (days)'],
            'carbon_footprint': material['Carbon Footprint (kg CO2/unit)'],
            'co2_emission_per_kg': material['CO2 Emission per kg (estimated)'],
            'load_handling_score': material['Load Handling Score'],
            'moisture_resistance': material['Moisture Resistance Score'],
            'thermal_resistance': material['Thermal Resistance Score'],
            'cost_per_unit_usd': material['Cost per Unit (USD)'],
            'supplier_region': material['Supplier Region'],
            'reusability_percent': material['Reusability (%)'],
            'recycled_content_percent': material['Recycled Content (%)'],
            'waste_reduction_impact': material['Waste Reduction Impact (%)'],
        }
        integrated_data.append(record)

integrated_df = pd.DataFrame(integrated_data)
print(f"✓ Created {len(integrated_df)} product-material combinations")



[3/6] Creating product-material combinations...
✓ Created 606000 product-material combinations


In [8]:
print("\n[4/6] Performing basic data cleaning...")

# Fill missing numeric values with median
numeric_columns = integrated_df.select_dtypes(include=[np.number]).columns
for col in numeric_columns:
    if integrated_df[col].isnull().any():
        integrated_df[col].fillna(integrated_df[col].median(), inplace=True)

# Fill missing categorical values with 'Unknown'
categorical_columns = integrated_df.select_dtypes(include=['object']).columns
for col in categorical_columns:
    if integrated_df[col].isnull().any():
        integrated_df[col].fillna('Unknown', inplace=True)

print("✓ Data cleaning complete")
print(f"  Missing values remaining: {integrated_df.isnull().sum().sum()}")



[4/6] Performing basic data cleaning...


C:\Users\sneha\AppData\Local\Temp\ipykernel_10380\1937448528.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  integrated_df[col].fillna(integrated_df[col].median(), inplace=True)
C:\Users\sneha\AppData\Local\Temp\ipykernel_10380\1937448528.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves

✓ Data cleaning complete
  Missing values remaining: 0


In [9]:
print("\n[5/6] Computing material-product compatibility...")

def calculate_compatibility_score(row):
    score = 0
    
    # Category match (40 points)
    if pd.notna(row['suitable_categories']):
        categories = str(row['suitable_categories']).lower()
        product_cat = str(row['product_category']).lower()
        if product_cat in categories:
            score += 40
    
    # Fragility match (30 points)
    fragility = row['fragility_index']
    if fragility >= 4:  # High fragility
        if row['load_handling_score'] >= 7:
            score += 30
        elif row['load_handling_score'] >= 5:
            score += 20
    elif fragility >= 2:  # Medium fragility
        if row['load_handling_score'] >= 5:
            score += 30
        elif row['load_handling_score'] >= 3:
            score += 20
    else:  # Low fragility
        score += 25
    
    # Weight capacity (30 points)
    weight = row['product_weight_kg']
    if weight < 0.5 and row['load_handling_score'] >= 3:
        score += 30
    elif weight < 2.0 and row['load_handling_score'] >= 5:
        score += 30
    elif row['load_handling_score'] >= 7:
        score += 30
    else:
        score += 15
    
    return min(score, 100)

integrated_df['compatibility_score'] = integrated_df.apply(calculate_compatibility_score, axis=1)
print(f"✓ Compatibility scores computed")
print(f"  Average score: {integrated_df['compatibility_score'].mean():.2f}")



[5/6] Computing material-product compatibility...
✓ Compatibility scores computed
  Average score: 58.82


In [10]:
print("\n[6/6] Saving integrated dataset...")

output_file = f"C:\\Users\\sneha\\Desktop\\ecopackai\\Dataset\\integrated_dataset.csv"
integrated_df.to_csv(output_file, index=False)
print(f"✓ Saved to: {output_file}")

# Save summary
summary_file = f"C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\docs\\integration_summary.textt"
with open(summary_file, 'w') as f:
    f.write("EcoPackAI - Data Integration Summary\n")
    f.write("=" * 60 + "\n")
    f.write(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
    f.write(f"Products: {len(products_df)}\n")
    f.write(f"Materials: {len(materials_df)}\n")
    f.write(f"Total combinations: {len(integrated_df)}\n\n")
    f.write(f"Dataset shape: {integrated_df.shape}\n")
    f.write(f"Missing values: {integrated_df.isnull().sum().sum()}\n\n")
    f.write("Columns:\n")
    for col in integrated_df.columns:
        f.write(f"  - {col}\n")

print(f"✓ Summary saved to: {summary_file}")



[6/6] Saving integrated dataset...
✓ Saved to: C:\Users\sneha\Desktop\ecopackai\Dataset\integrated_dataset.csv
✓ Summary saved to: C:\Users\sneha\Desktop\ecopackai\Data\docs\integration_summary.textt


In [11]:
print(integrated_df.head(3).to_string())


   product_id   product_name product_category  product_weight_kg  fragility_index shipping_type material_id    material_type                            packaging_type                                                       suitable_categories  recyclability_percent recyclability_category  biodegradation_days  carbon_footprint  co2_emission_per_kg  load_handling_score  moisture_resistance  thermal_resistance  cost_per_unit_usd supplier_region  reusability_percent  recycled_content_percent  waste_reduction_impact  compatibility_score
0           1  Chocolate Box             Food                0.5                2           Air    MAT_0001        Cardboard                           Cardboard Boxes                      E-commerce, Food & Beverage, Consumer Goods, Apparel                     98                   High                188.0              0.78                 0.54                  6.0                  5.0                 4.0               2.24            EMEA                 49.0